In [6]:
import re

In [1]:
document1 = """
University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject. Students who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department. The department may consider the medical reason, but medical evidence does not automatically remove the attendance shortage. In special cases, the department can review the student's situation and make a decision according to university rules.

Students should arrive on time and avoid unnecessary absences. Attendance records are maintained by the department throughout the semester.
"""


In [2]:
document2 = """
Assignment Submission Policy

Students must submit their assignments through the university learning portal before the given deadline. Every assignment has a specific submission date announced by the teacher. Students should upload their work in the required file format and check that the uploaded file is correct.

Late assignments may receive a penalty according to the course policy. Students are responsible for checking the submission status after uploading their work. If the portal shows an error, students should inform their teacher or the IT help desk.

If a student faces a serious medical emergency or a technical problem with the university portal, the teacher may allow an extension after reviewing the evidence. However, an extension is not automatic, and students must contact the teacher as soon as possible.

Teachers should clearly explain assignment requirements, deadlines, marking criteria, and accepted file formats. Students should avoid waiting until the last minute to upload their assignments.
"""

In [3]:
document3 = """
University IT Help Desk

The university IT help desk provides technical support to students and staff. Students can contact the help desk if they have problems with their university accounts, learning portal, email, or computer systems. The help desk normally operates from Monday to Saturday, between 9 AM and 5 PM.

Students should provide their registration number and a clear explanation of the problem when requesting support. For password problems, students can use the password recovery option on the login page. If the recovery option does not work, they should contact the IT help desk for further assistance.

The help desk usually responds to normal requests during working hours. However, urgent problems that affect online examinations or important university services may receive priority, even when the normal support schedule is busy.

Students should never share their passwords with other people. Staff members are responsible for protecting student account information and maintaining the university's computer systems.
"""

In [4]:
print("Document 1 Words:", len(document1.split()))
print("Document 2 Words:", len(document2.split()))
print("Document 3 Words:", len(document3.split()))

Document 1 Words: 153
Document 2 Words: 152
Document 3 Words: 155


In [5]:
Documents = [
    document1,
    document2,
    document3
]
print(len(Documents))

3


In [8]:
def word_location(text):
    return [(m.start(),m.end()) for m in re.finditer(r"\S+",text)]

In [9]:
print(word_location("My university is here"))

[(0, 2), (3, 13), (14, 16), (17, 21)]


In [35]:
def sentence_location(text):
    sentences = []

    for m in re.finditer(r'[^.!?]+[.!?]?', text):
        sentences.append((m.start(), m.end()))

    return sentences

In [36]:
print(sentence_location("My university is here. I study AI."))

[(0, 22), (22, 34)]


In [72]:
def create_chunks(text, size=40, overlap=0):
    position = word_location(text)
    chunks = []

    if overlap < 0 or overlap >= size:
        raise ValueError("Invalid overlap")
    step = size - overlap

    for i in range(0, len(position), step):
        start = position[i][0]
        end = position[min(i + size, len(position)) - 1][1]

        chunks.append({
            "id": len(chunks) + 1,
            "start": start,
            "end": end,
            "text": text[start:end]
        })
        if i + size >= len(position):
            break

    return chunks

In [33]:
chunks = create_chunks(document1, 40, 0)

print("Total Chunks:", len(chunks))

for chunk in chunks:
    print("ID:", chunk["id"])
    print("Start:", chunk["start"])
    print("End:", chunk["end"])
    print("Text:", chunk["text"])
    print("----------------")

Total Chunks: 4
ID: 1
Start: 1
End: 298
Text: University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject. Students
----------------
ID: 2
Start: 299
End: 559
Text: who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their
----------------
ID: 3
Start: 560
End: 828
Text: teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department. The department may consider the medical reason, but medical evidence does not automatically remove the attendance shortage. In special cases, the
----------------
ID: 4
Start

In [34]:
chunks = create_chunks(document1, 40, 10)

print("Total Chunks:", len(chunks))

for chunk in chunks:
    print("ID:", chunk["id"])
    print("Start:", chunk["start"])
    print("End:", chunk["end"])
    print("Text:", chunk["text"])
    print("----------------")

Total Chunks: 5
ID: 1
Start: 1
End: 298
Text: University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject. Students
----------------
ID: 2
Start: 233
End: 491
Text: maintain at least 75 percent attendance in each subject. Students who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class. Students should check
----------------
ID: 3
Start: 421
End: 686
Text: for recording student attendance in every class. Students should check their attendance records regularly and report any mistakes to their teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department. The
----------------
ID: 4
Start: 618

In [66]:
def sentence_chunks(text, budget=40):
    sentences = sentence_location(text)
    chunks = []
    current_chunk = ""
    count = 0
    chunk_start = None
    chunk_end = None

    for start, end in sentences:
        sentence = text[start:end]
        words = len(sentence.split())

        if words > budget:
            if current_chunk:
                chunks.append({
                    "id": len(chunks) + 1,
                    "start": chunk_start,
                    "end": chunk_end,
                    "text": current_chunk
                })

            current_chunk = ""
            count = 0
            chunk_start = None
            chunk_end = None

            word_spans = word_location(sentence)

            for i in range(0, len(word_spans), budget):
                s = word_spans[i][0]
                e = word_spans[min(i + budget, len(word_spans)) - 1][1]

                chunks.append({
                    "id": len(chunks) + 1,
                    "start": start + s,
                    "end": start + e,
                    "text": sentence[s:e]
                })

            continue

        chunk_start = start if chunk_start is None else chunk_start

        if count + words <= budget:
            current_chunk += sentence
            chunk_end = end
            count += words
        else:
            if current_chunk:
                chunks.append({
                    "id": len(chunks) + 1,
                    "start": chunk_start,
                    "end": chunk_end,
                    "text": current_chunk
                })

            current_chunk = sentence
            count = words
            chunk_start = start
            chunk_end = end

    if current_chunk:
        chunks.append({
            "id": len(chunks) + 1,
            "start": chunk_start,
            "end": chunk_end,
            "text": current_chunk
        })

    return chunks

In [58]:
chunks = sentence_chunks(document1, 40)

print("Total Sentence Chunks:", len(chunks))

for chunk in chunks:
    print("ID:", chunk["id"])
    print("Start:", chunk["start"])
    print("End:", chunk["end"])
    print("Words:", len(chunk["text"].split()))
    print("Text:", chunk["text"])
    print("-------------------")

Total Sentence Chunks: 5
ID: 1
Start: 0
End: 289
Words: 39
Text: 
University Attendance Policy

Students are expected to attend all scheduled classes regularly. Attendance is important because it helps students understand lectures and complete their coursework. The university requires students to maintain at least 75 percent attendance in each subject.
-------------------
ID: 2
Start: 289
End: 469
Words: 28
Text:  Students who have less than 75 percent attendance may not be allowed to appear in the final examination.

Teachers are responsible for recording student attendance in every class.
-------------------
ID: 3
Start: 469
End: 682
Words: 32
Text:  Students should check their attendance records regularly and report any mistakes to their teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department.
-------------------
ID: 4
Start: 682
End: 925
Words: 35
Text:  The department may consider the medical reason, but medical evi

In [67]:
long_sentence = " ".join(["University"] * 50) + "."

long_chunks = sentence_chunks(long_sentence, 40)

for chunk in long_chunks:
    print("Words:", len(chunk["text"].split()))
    print("Text:", chunk["text"])

Words: 40
Text: University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University University
Words: 10
Text: University University University University University University University University University University.


In [69]:
empty_chunks = sentence_chunks("", 40)
print(empty_chunks)

short_chunks = sentence_chunks("AI is useful. It helps students.", 40)

for chunk in short_chunks:
    print(chunk)

[]
{'id': 1, 'start': 0, 'end': 32, 'text': 'AI is useful. It helps students.'}


In [74]:
create_chunks(document1, 40, 40)

ValueError: Invalid overlap

In [76]:
for document in Documents:
    print("Overlap 0:", len(create_chunks(document, 40, 0)))
    print("Overlap 10:", len(create_chunks(document, 40, 10)))
    print("Sentence Aware:", len(sentence_chunks(document, 40)))
    print("-------------------")

Overlap 0: 4
Overlap 10: 5
Sentence Aware: 5
-------------------
Overlap 0: 4
Overlap 10: 5
Sentence Aware: 6
-------------------
Overlap 0: 4
Overlap 10: 5
Sentence Aware: 5
-------------------


In [77]:
c0 = create_chunks(document1, 40, 0)
c10 = create_chunks(document1, 40, 10)

print("Overlap 0:", sum(len(c["text"]) for c in c0))
print("Overlap 10:", sum(len(c["text"]) for c in c10))

Overlap 0: 1062
Overlap 10: 1333


In [78]:
for chunk in sentence_chunks(document1, 40):
    if "medical evidence" in chunk["text"]:
        print("Chunk ID:", chunk["id"])
        print(chunk["text"])

Chunk ID: 3
 Students should check their attendance records regularly and report any mistakes to their teachers.

If a student is absent because of illness, the student should provide valid medical evidence to the department.
Chunk ID: 4
 The department may consider the medical reason, but medical evidence does not automatically remove the attendance shortage. In special cases, the department can review the student's situation and make a decision according to university rules.


In [79]:
for chunk in sentence_chunks(document2, 40):
    if "extension" in chunk["text"]:
        print("Chunk ID:", chunk["id"])
        print(chunk["text"])
        print("----------------")

Chunk ID: 4


If a student faces a serious medical emergency or a technical problem with the university portal, the teacher may allow an extension after reviewing the evidence.
----------------
Chunk ID: 5
 However, an extension is not automatic, and students must contact the teacher as soon as possible.

Teachers should clearly explain assignment requirements, deadlines, marking criteria, and accepted file formats.
----------------


In [80]:
for chunk in sentence_chunks(document3, 40):
    if "urgent" in chunk["text"]:
        print("Chunk ID:", chunk["id"])
        print(chunk["text"])

Chunk ID: 4


The help desk usually responds to normal requests during working hours. However, urgent problems that affect online examinations or important university services may receive priority, even when the normal support schedule is busy.


In [81]:
text = "Dr. Ahmed teaches AI. Mr. Ali studies CS."

print(sentence_location(text))

[(0, 3), (3, 21), (21, 25), (25, 41)]
